# NB5 — Type-aware Pairwise V1: S1 smoke + S2.5 + S3/S3.1

Notebook wrapper cho frozen Core-7 V2 scorer data. Core logic nằm trong `src/scorer/`.
Mục tiêu: verify artifacts → S1 smoke → S2 model checks → S2.5 tiny overfit → S3 full baseline.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
# Khi test PR, đặt FASHION_REPO_REF=feat/scorer-s3-full-baseline.
REPO_REF = os.environ.get("FASHION_REPO_REF", "main")
REPO_ROOT = Path.cwd() / "opisoverated"

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "checkout", "--detach", "FETCH_HEAD"],
        check=True,
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

commit = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()
print("REPO_ROOT:", REPO_ROOT)
print("REPO_REF :", REPO_REF)
print("Git commit:", commit)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ARTIFACT_ROOT = Path(
    os.environ.get("FASHION_ARTIFACT_ROOT", "/content/drive/MyDrive/ML_Final")
)
os.environ["FASHION_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["FASHION_EMBEDDING_CACHE"] = str(
    ARTIFACT_ROOT / "fashionclip_item_embeddings.pt"
)
os.environ["FASHION_EMBEDDING_MANIFEST"] = str(
    ARTIFACT_ROOT / "embedding_manifest_v1.json"
)
os.environ["FASHION_CORE7_DIR"] = str(
    ARTIFACT_ROOT / "polyvore_core7_v2" / "core7_drop_v2"
)
os.environ["FASHION_SCORER_READY_DIR"] = str(
    ARTIFACT_ROOT / "polyvore_core7_v2" / "scorer_ready_v2"
)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)


In [ ]:
%pip install -q pyyaml

import json
import yaml
from src.data.runtime_paths import load_runtime_paths
from src.data.build_core7_scorer_dataset import sha256_file

CONFIG_PATH = REPO_ROOT / "configs/scorer_type_aware_pairwise_v1.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

assert config["model"]["name"] == "type_aware_pairwise_v1"
assert config["model"]["embedding_dim"] == 512
assert config["model"]["category_vocab_size"] == 8
assert config["model"]["category_embedding_init_std"] == 0.02
assert config["data"]["min_items"] == 3
assert config["data"]["max_items"] == 8

paths = load_runtime_paths(repo_root=REPO_ROOT)
print("Embedding cache :", paths.embedding_cache)
print("Manifest        :", paths.embedding_manifest)
print("Core-7 dir      :", paths.core7_dir)
print("Scorer-ready dir:", paths.scorer_ready_dir)
print("CONFIG: PASS")


In [ ]:
required = [
    paths.embedding_cache,
    paths.embedding_manifest,
    paths.scorer_ready_dir / "dataset_manifest_v2.json",
    paths.scorer_ready_dir / "split_manifest_v2.json",
    paths.scorer_ready_dir / "final_validation_v2.json",
]
for split in ("train", "valid", "test"):
    required += [
        paths.scorer_ready_dir / f"scorer_ready_v2_{split}.jsonl",
        paths.core7_dir / f"core7_item_metadata_v1_{split}.jsonl",
    ]

missing = [path for path in required if not path.is_file()]
assert not missing, f"Missing artifacts: {missing}"

with (REPO_ROOT / "artifacts/data_v2_reference.json").open("r", encoding="utf-8") as f:
    reference = json.load(f)
freeze = reference["freeze_fields_after_rebuild"]

assert reference["status"] == "READY_TO_TRAIN"
assert reference["dataset_version"] == "polyvore1000-core7-compat-v2"

checks = {
    "embedding_cache": (paths.embedding_cache, freeze["embedding_cache_sha256"]),
    "embedding_manifest": (paths.embedding_manifest, freeze["embedding_manifest_sha256"]),
    "category_mapping": (
        REPO_ROOT / "configs/category_mapping_core7_v2.json",
        freeze["mapping_sha256"],
    ),
}
for split in ("train", "valid", "test"):
    checks[split] = (
        paths.scorer_ready_dir / f"scorer_ready_v2_{split}.jsonl",
        freeze[f"{split}_sha256"],
    )

for name, (path, expected) in checks.items():
    actual = sha256_file(path)
    print(name, "PASS" if actual == expected else "FAIL")
    assert actual == expected, f"{name} hash mismatch"

print("FROZEN V2 HASHES: PASS")


In [ ]:
test_files = [
    "test_scorer_dataset.py",
    "test_pair_generation.py",
    "test_scorer_metrics.py",
    "test_scorer_model.py",
    "test_scorer_training.py",
    "test_scorer_checkpoint.py",
    "test_scorer_full_training.py",
]
for pattern in test_files:
    subprocess.run(
        [
            sys.executable, "-m", "unittest", "discover",
            "-s", "tests", "-p", pattern, "-v",
        ],
        cwd=REPO_ROOT,
        check=True,
    )
print("S1/S2/TRAINING/CHECKPOINT UNIT TESTS: PASS")


In [ ]:
import torch
from functools import partial
from torch.utils.data import DataLoader, Subset
from src.scorer.dataset import (
    EmbeddingStore,
    build_dataset_from_runtime,
    collate_scorer_batch,
    flatten_family_indices,
)

embedding_store = EmbeddingStore(paths.embedding_cache)
train_dataset = build_dataset_from_runtime(
    paths,
    "train",
    embedding_store=embedding_store,
)

assert len(train_dataset) == freeze["train_sample_count"]
assert len(train_dataset.pair_families) * 2 == len(train_dataset)

SMOKE_FAMILIES = 128
families = train_dataset.pair_families[:SMOKE_FAMILIES]
smoke_indices = flatten_family_indices(families)
smoke_dataset = Subset(train_dataset, smoke_indices)

collate_fn = partial(
    collate_scorer_batch,
    max_items=config["data"]["max_items"],
)
smoke_loader = DataLoader(
    smoke_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)
batch = next(iter(smoke_loader))

B = len(batch["sample_ids"])
assert batch["item_embeddings"].shape == (B, 8, 512)
assert batch["coarse_category_ids"].shape == (B, 8)
assert batch["item_mask"].shape == (B, 8)
assert batch["pair_mask"].shape == (B, 8, 8)
assert batch["labels"].shape == (B,)
assert torch.isfinite(batch["item_embeddings"]).all()

item_counts = batch["item_mask"].sum(dim=1)
assert item_counts.min() >= 3 and item_counts.max() <= 8
assert torch.all(batch["item_embeddings"][~batch["item_mask"]] == 0)
assert torch.all(batch["coarse_category_ids"][~batch["item_mask"]] == 0)

actual_pairs = batch["pair_mask"].sum(dim=(1, 2))
expected_pairs = item_counts * (item_counts - 1) // 2
assert torch.equal(actual_pairs, expected_pairs)
assert actual_pairs.max() <= 28

print("Train samples :", len(train_dataset))
print("Pair families :", len(train_dataset.pair_families))
print("Smoke samples :", len(smoke_dataset))
print("BATCH + MASK: PASS")


In [ ]:
pos_idx, neg_idx = families[0]
pos_record = train_dataset.records[pos_idx]
neg_record = train_dataset.records[neg_idx]

assert pos_record["label"] == 1
assert neg_record["label"] == 0
assert neg_record["paired_positive_sample_id"] == pos_record["sample_id"]

differences = [
    i for i, (p, n) in enumerate(zip(pos_record["items"], neg_record["items"]))
    if p != n
]
assert len(differences) == 1

swap_index = differences[0]
original_item = pos_record["items"][swap_index]
replacement_item = neg_record["items"][swap_index]
original_meta = train_dataset.metadata_by_item[original_item]
replacement_meta = train_dataset.metadata_by_item[replacement_item]
assert original_meta["master_category"] == replacement_meta["master_category"]

pos_sample = train_dataset[pos_idx]
first_item_id = pos_sample["item_ids"][0]
cache_row = train_dataset.embedding_row_by_item[first_item_id]
assert torch.allclose(
    pos_sample["item_embeddings"][0],
    train_dataset.embedding_matrix[cache_row].float(),
)
print("LOOKUPS + PAIRING: PASS")


In [ ]:
from src.scorer.evaluate import evaluate_predictions

smoke_records = [train_dataset.records[index] for index in smoke_indices]
labels = [row["label"] for row in smoke_records]
metrics = evaluate_predictions(
    sample_ids=[row["sample_id"] for row in smoke_records],
    paired_positive_sample_ids=[
        row["paired_positive_sample_id"] for row in smoke_records
    ],
    labels=labels,
    logits=[1.0 if label == 1 else -1.0 for label in labels],
)

assert metrics["roc_auc"] == 1.0
assert metrics["fitb_2way"] == 1.0
assert metrics["mean_logit_margin"] == 2.0
assert metrics["median_logit_margin"] == 2.0
assert metrics["sample_count"] == 256
assert metrics["paired_family_count"] == 128

s1_report = {
    "frozen_artifacts": "PASS",
    "unit_tests": "PASS",
    "embedding_lookup": "PASS",
    "category_lookup": "PASS",
    "padding": "PASS",
    "item_mask": "PASS",
    "pair_mask": "PASS",
    "positive_negative_pairing": "PASS",
    "evaluator": "PASS",
    "smoke_sample_count": len(smoke_dataset),
}
print(metrics)
print(s1_report)
print("=" * 72)
print("S1 SCORER SMOKE TEST: PASS")
print("=" * 72)


## S2.5 — Tiny-set overfit

Sanity run trên đúng 32 complete positive-negative families (64 samples). Đây không phải benchmark và không dùng test split.


In [ ]:
from src.scorer.model import TypeAwarePairwiseScorer
from src.scorer.train import (
    build_tiny_overfit_loader,
    resolve_device,
    run_tiny_overfit,
    set_reproducible_seed,
)

TINY_FAMILIES = 32
TINY_MAX_EPOCHS = 300
TINY_TARGET_ROC_AUC = 0.99
TINY_TARGET_FITB = 0.99
TINY_MAX_LOSS_RATIO = 0.25
SEED = int(config["training"]["seed"])

set_reproducible_seed(SEED)
tiny_loader, tiny_selection = build_tiny_overfit_loader(
    train_dataset,
    family_count=TINY_FAMILIES,
    batch_size=2 * TINY_FAMILIES,
    seed=SEED,
    max_items=config["data"]["max_items"],
)
model = TypeAwarePairwiseScorer.from_config(config)
device = resolve_device()
print("S2.5 device:", device)
print("S2.5 selection:", tiny_selection)

tiny_result = run_tiny_overfit(
    model,
    tiny_loader,
    config,
    device=device,
    max_epochs=TINY_MAX_EPOCHS,
    expected_family_count=TINY_FAMILIES,
    target_roc_auc=TINY_TARGET_ROC_AUC,
    target_fitb=TINY_TARGET_FITB,
    max_loss_ratio=TINY_MAX_LOSS_RATIO,
    use_amp=False,
)
print("Initial:", tiny_result["initial"])
print("Final  :", tiny_result["final"])
print("Epochs :", tiny_result["epochs_ran"])


In [ ]:
SCORER_RUN_ROOT = Path(
    os.environ.get(
        "FASHION_SCORER_RUN_ROOT",
        "/content/drive/MyDrive/scorer_runs",
    )
)
S2_5_RUN_DIR = SCORER_RUN_ROOT / "type_aware_pairwise_v1" / commit[:12]
S2_5_RUN_DIR.mkdir(parents=True, exist_ok=True)
S2_5_REPORT_PATH = S2_5_RUN_DIR / "s2_5_tiny_overfit_report.json"

s2_5_report = {
    "scorer_version": "type_aware_pairwise_v1",
    "dataset_version": reference["dataset_version"],
    "category_mapping_version": reference["category_mapping_version"],
    "negative_protocol_version": reference["negative_protocol_version"],
    "embedding_version": reference["embedding_version"],
    "git_commit": commit,
    "frozen_input_sha256": {
        "mapping": freeze["mapping_sha256"],
        "embedding_cache": freeze["embedding_cache_sha256"],
        "embedding_manifest": freeze["embedding_manifest_sha256"],
        "train": freeze["train_sha256"],
    },
    "selection": tiny_selection,
    **tiny_result,
}
with S2_5_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(s2_5_report, f, ensure_ascii=False, indent=2)

print("S2.5 report:", S2_5_REPORT_PATH)
if tiny_result["status"] != "PASS":
    raise RuntimeError(
        "S2.5 FAIL: debug labels/lookups/masks/pairs/logits/BCE/optimizer/gradients; "
        "do not start S3."
    )
print("=" * 72)
print("S2.5 TINY OVERFIT: PASS")
print("=" * 72)


## S3 — Full baseline training

Train toàn bộ train split, đánh giá valid sau mỗi epoch, chọn `best.pt` chỉ bằng validation ROC-AUC và dừng sớm với patience 5. Test split tuyệt đối không đi vào runner này.

Đổi `RUN_S3_FULL = True` khi sẵn sàng chạy. Nếu Colab ngắt giữa chừng, chạy lại với `RESUME_S3 = True` để tiếp tục từ `last.pt`.


In [ ]:
RUN_S3_FULL = False  # Đổi thành True để bắt đầu full training.
RESUME_S3 = False    # Chỉ đổi True khi tiếp tục một run bị ngắt.
S3_NUM_WORKERS = 0   # 0 là lựa chọn an toàn khi đọc artifact từ Drive.

S3_RUN_DIR = (
    SCORER_RUN_ROOT
    / "type_aware_pairwise_v1"
    / commit[:12]
    / f"seed_{SEED}"
    / "s3_full_baseline"
)
print("S3 output :", S3_RUN_DIR)
print("Run full  :", RUN_S3_FULL)
print("Resume    :", RESUME_S3)
if not RUN_S3_FULL:
    print("S3 chưa chạy. Đổi RUN_S3_FULL thành True rồi chạy lại cell này và hai cell dưới.")


In [ ]:
from src.scorer.checkpoint import canonical_provenance, capture_git_state
from src.scorer.train import build_full_training_loaders, run_full_training

s3_summary = None
if RUN_S3_FULL:
    if "tiny_result" in globals():
        s2_5_status = tiny_result["status"]
    elif S2_5_REPORT_PATH.is_file():
        with S2_5_REPORT_PATH.open("r", encoding="utf-8") as f:
            s2_5_status = json.load(f)["status"]
    else:
        raise RuntimeError("Thiếu S2.5 PASS report của đúng Git commit; chưa được chạy S3.")
    assert s2_5_status == "PASS", "S2.5 chưa PASS; không được full-train."

    valid_dataset = build_dataset_from_runtime(
        paths,
        "valid",
        embedding_store=embedding_store,
    )
    assert len(train_dataset) == freeze["train_sample_count"]
    assert len(valid_dataset) == freeze["valid_sample_count"]

    set_reproducible_seed(SEED)
    train_loader, valid_loader = build_full_training_loaders(
        train_dataset,
        valid_dataset,
        config,
        num_workers=S3_NUM_WORKERS,
    )

    dataset_manifest_path = paths.scorer_ready_dir / "dataset_manifest_v2.json"
    dataset_manifest_sha256 = sha256_file(dataset_manifest_path)
    embedding_manifest_sha256 = sha256_file(paths.embedding_manifest)
    assert embedding_manifest_sha256 == freeze["embedding_manifest_sha256"]
    provenance = canonical_provenance(
        dataset_manifest_sha256=dataset_manifest_sha256,
        embedding_manifest_sha256=embedding_manifest_sha256,
    )
    for key in (
        "dataset_version",
        "category_mapping_version",
        "negative_protocol_version",
        "embedding_version",
    ):
        assert provenance[key] == reference[key], f"{key} mismatch"

    git_state = capture_git_state(REPO_ROOT)
    assert git_state["git_commit"] == commit
    assert git_state["git_tree_clean"] is True, "Official S3 requires clean Git tree"

    set_reproducible_seed(SEED)
    s3_model = TypeAwarePairwiseScorer.from_config(config)
    s3_device = resolve_device()

    def print_epoch(row):
        print(
            f"epoch={row['epoch']:02d} "
            f"train_loss={row['train_loss']:.6f} "
            f"valid_loss={row['valid_loss']:.6f} "
            f"auc={row['valid_roc_auc']:.6f} "
            f"fitb={row['valid_fitb_2way']:.6f} "
            f"improved={row['improved']} "
            f"patience={row['epochs_without_improvement']}/{config['training']['early_stopping_patience']}"
        )

    print("S3 device :", s3_device)
    print("Train rows:", len(train_dataset))
    print("Valid rows:", len(valid_dataset))
    s3_summary = run_full_training(
        s3_model,
        train_loader,
        valid_loader,
        config,
        output_dir=S3_RUN_DIR,
        provenance=provenance,
        git_state=git_state,
        device=s3_device,
        resume=RESUME_S3,
        epoch_callback=print_epoch,
    )


In [ ]:
if globals().get("s3_summary") is not None:
    print("=" * 72)
    print("S3 FULL BASELINE:", s3_summary["status"])
    print("Epochs completed :", s3_summary["epochs_completed"])
    print("Best epoch       :", s3_summary["best_epoch"])
    print("Best valid AUC   :", s3_summary["best_valid_roc_auc"])
    best_metrics = s3_summary["best_validation_metrics"]
    print("Best valid FITB  :", best_metrics["fitb_2way"])
    print("Best mean margin :", best_metrics["mean_logit_margin"])
    print("Best median margin:", best_metrics["median_logit_margin"])
    print("Valid samples    :", best_metrics["sample_count"])
    print("Paired families  :", best_metrics["paired_family_count"])
    print("best.pt          :", s3_summary["output_paths"]["best_checkpoint"])
    print("last.pt          :", s3_summary["output_paths"]["last_checkpoint"])
    print("Training history :", s3_summary["output_paths"]["training_history"])
    print("=" * 72)
else:
    print("S3 chưa được chạy; không có kết quả để tổng kết.")


In [ ]:
if globals().get("s3_summary") is not None:
    import matplotlib.pyplot as plt

    epochs = [row["epoch"] for row in s3_summary["history"]]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, [row["train_loss"] for row in s3_summary["history"]], label="train loss")
    axes[0].plot(epochs, [row["valid_loss"] for row in s3_summary["history"]], label="valid loss")
    axes[0].set_xlabel("epoch")
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(epochs, [row["valid_roc_auc"] for row in s3_summary["history"]], label="valid ROC-AUC")
    axes[1].plot(epochs, [row["valid_fitb_2way"] for row in s3_summary["history"]], label="valid FITB")
    axes[1].axvline(s3_summary["best_epoch"], color="black", linestyle="--", label="best epoch")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_title("Validation metrics")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


## S3-control và S3.1 — AMP-off / paired ranking

Giữ nguyên Frozen Data V2 và S3 baseline. Control chỉ tắt AMP. S3.1 giữ AMP như baseline nhưng train theo complete family bằng `BCE + 0.5 × paired logistic ranking loss`. Hai experiment ghi vào output directory riêng và không đọc test split.


In [ ]:
AMP_CONTROL_CONFIG_PATH = REPO_ROOT / "configs/scorer_type_aware_pairwise_v1_amp_off_control.yaml"
PAIRED_CONFIG_PATH = REPO_ROOT / "configs/scorer_type_aware_pairwise_v1_paired_ranking_v1.yaml"
with AMP_CONTROL_CONFIG_PATH.open("r", encoding="utf-8") as f:
    amp_control_config = yaml.safe_load(f)
with PAIRED_CONFIG_PATH.open("r", encoding="utf-8") as f:
    paired_config = yaml.safe_load(f)

assert amp_control_config["model"] == config["model"]
assert amp_control_config["data"] == config["data"]
assert amp_control_config["training"]["mixed_precision"] is False
assert amp_control_config["training"]["objective"] == "bce"
assert paired_config["model"] == config["model"]
assert paired_config["data"] == config["data"]
assert paired_config["training"]["objective"] == "bce_plus_paired_logistic"
assert paired_config["training"]["paired_batching"] is True
assert paired_config["training"]["paired_ranking_weight"] == 0.5

RUN_S3_AMP_OFF_CONTROL = False
RESUME_S3_AMP_OFF_CONTROL = False
RUN_S3_1_PAIRED = False
RESUME_S3_1_PAIRED = False

from src.scorer.train import build_paired_training_loaders

def prepare_s3_experiment(experiment_config, *, paired):
    if "tiny_result" in globals():
        status = tiny_result["status"]
    elif S2_5_REPORT_PATH.is_file():
        with S2_5_REPORT_PATH.open("r", encoding="utf-8") as f:
            status = json.load(f)["status"]
    else:
        raise RuntimeError("Thiếu S2.5 PASS report của đúng Git commit.")
    assert status == "PASS", "S2.5 chưa PASS."

    experiment_valid_dataset = build_dataset_from_runtime(
        paths, "valid", embedding_store=embedding_store
    )
    assert len(train_dataset) == freeze["train_sample_count"]
    assert len(experiment_valid_dataset) == freeze["valid_sample_count"]

    seed = int(experiment_config["training"]["seed"])
    set_reproducible_seed(seed)
    loader_builder = (
        build_paired_training_loaders if paired else build_full_training_loaders
    )
    experiment_train_loader, experiment_valid_loader = loader_builder(
        train_dataset,
        experiment_valid_dataset,
        experiment_config,
        num_workers=S3_NUM_WORKERS,
    )

    dataset_manifest_sha256 = sha256_file(
        paths.scorer_ready_dir / "dataset_manifest_v2.json"
    )
    embedding_manifest_sha256 = sha256_file(paths.embedding_manifest)
    assert embedding_manifest_sha256 == freeze["embedding_manifest_sha256"]
    experiment_provenance = canonical_provenance(
        dataset_manifest_sha256=dataset_manifest_sha256,
        embedding_manifest_sha256=embedding_manifest_sha256,
    )
    for key in ("dataset_version", "category_mapping_version", "negative_protocol_version", "embedding_version"):
        assert experiment_provenance[key] == reference[key], f"{key} mismatch"

    experiment_git_state = capture_git_state(REPO_ROOT)
    assert experiment_git_state["git_commit"] == commit
    assert experiment_git_state["git_tree_clean"] is True

    set_reproducible_seed(seed)
    experiment_model = TypeAwarePairwiseScorer.from_config(experiment_config)
    return (
        experiment_model,
        experiment_train_loader,
        experiment_valid_loader,
        experiment_provenance,
        experiment_git_state,
        resolve_device(),
    )

def run_s3_experiment(experiment_config, output_name, *, paired, resume):
    model_, train_loader_, valid_loader_, provenance_, git_state_, device_ = (
        prepare_s3_experiment(experiment_config, paired=paired)
    )
    seed = int(experiment_config["training"]["seed"])
    output_dir = (
        SCORER_RUN_ROOT / "type_aware_pairwise_v1" / commit[:12]
        / f"seed_{seed}" / output_name
    )

    def print_experiment_epoch(row):
        print(
            f"epoch={row['epoch']:02d} total={row['train_loss']:.6f} "
            f"bce={row['train_bce_loss']:.6f} rank={row['train_paired_ranking_loss']:.6f} "
            f"valid_bce={row['valid_loss']:.6f} auc={row['valid_roc_auc']:.6f} "
            f"fitb={row['valid_fitb_2way']:.6f} mean_margin={row['valid_mean_logit_margin']:.6f} "
            f"patience={row['epochs_without_improvement']}/{experiment_config['training']['early_stopping_patience']}"
        )

    print("Experiment:", experiment_config["experiment"]["name"])
    print("Device    :", device_)
    print("Output    :", output_dir)
    return run_full_training(
        model_, train_loader_, valid_loader_, experiment_config,
        output_dir=output_dir, provenance=provenance_, git_state=git_state_,
        device=device_, resume=resume, epoch_callback=print_experiment_epoch,
    )


In [ ]:
amp_control_summary = None
paired_summary = None
if RUN_S3_AMP_OFF_CONTROL:
    amp_control_summary = run_s3_experiment(
        amp_control_config,
        "s3_amp_off_control",
        paired=False,
        resume=RESUME_S3_AMP_OFF_CONTROL,
    )
if RUN_S3_1_PAIRED:
    paired_summary = run_s3_experiment(
        paired_config,
        "s3_1_paired_ranking_v1",
        paired=True,
        resume=RESUME_S3_1_PAIRED,
    )
if not RUN_S3_AMP_OFF_CONTROL and not RUN_S3_1_PAIRED:
    print("Chọn đúng một RUN flag = True để chạy experiment.")


In [ ]:
BASELINE_AUC = 0.5818041289285704
BASELINE_FITB = 0.6234676007005254

def print_full_metric_summary(label, summary):
    if summary is None:
        return
    metrics = summary["best_validation_metrics"]
    print("=" * 72)
    print(label, summary["status"])
    print("Best epoch        :", summary["best_epoch"])
    print("ROC-AUC           :", metrics["roc_auc"])
    print("2-way FITB        :", metrics["fitb_2way"])
    print("Mean logit margin :", metrics["mean_logit_margin"])
    print("Median margin     :", metrics["median_logit_margin"])
    print("Samples           :", metrics["sample_count"])
    print("Paired families   :", metrics["paired_family_count"])
    print("AUC delta vs S3   :", metrics["roc_auc"] - BASELINE_AUC)
    print("FITB delta vs S3  :", metrics["fitb_2way"] - BASELINE_FITB)
    print("best.pt           :", summary["output_paths"]["best_checkpoint"])

print_full_metric_summary("S3 AMP-OFF CONTROL", amp_control_summary)
print_full_metric_summary("S3.1 PAIRED RANKING", paired_summary)


## S3.1 control — paired ranking, balanced category init, 60 epochs

Run mới từ đầu để kiểm tra S3.1 cũ khi đổi `category_embedding_init_std` thành `1 / sqrt(32)` và tăng trần từ 30 lên 60 epoch. Objective, paired batching, ranking weight, AMP, optimizer, batch size và seed vẫn giữ nguyên. Output riêng, không overwrite run S3.1 cũ và không đọc test split.


In [ ]:
import math

PAIRED_BALANCED_60_CONFIG_PATH = (
    REPO_ROOT / "configs/scorer_type_aware_pairwise_v1_paired_ranking_balanced_60ep.yaml"
)
with PAIRED_BALANCED_60_CONFIG_PATH.open("r", encoding="utf-8") as f:
    paired_balanced_60_config = yaml.safe_load(f)

expected_std = 1.0 / math.sqrt(32.0)
actual_std = float(paired_balanced_60_config["model"]["category_embedding_init_std"])
assert math.isclose(actual_std, expected_std, rel_tol=0.0, abs_tol=1e-15)
for key, value in paired_config["model"].items():
    if key != "category_embedding_init_std":
        assert paired_balanced_60_config["model"][key] == value, key
for key, value in paired_config["training"].items():
    if key != "max_epochs":
        assert paired_balanced_60_config["training"][key] == value, key
assert paired_balanced_60_config["data"] == paired_config["data"]
assert paired_balanced_60_config["training"]["objective"] == "bce_plus_paired_logistic"
assert paired_balanced_60_config["training"]["paired_batching"] is True
assert paired_balanced_60_config["training"]["paired_ranking_weight"] == 0.5
assert paired_balanced_60_config["training"]["mixed_precision"] is True
assert paired_balanced_60_config["training"]["max_epochs"] == 60
assert paired_balanced_60_config["training"]["seed"] == 42

RUN_S3_1_PAIRED_BALANCED_60 = False  # Đổi True để bắt đầu run mới từ epoch 1.
RESUME_S3_1_PAIRED_BALANCED_60 = False  # Chỉ True sau khi runtime bị ngắt giữa run này.

print("Config             :", PAIRED_BALANCED_60_CONFIG_PATH)
print("Category init std  :", actual_std)
print("Expected cat norm  :", math.sqrt(32.0) * actual_std)
print("Max epochs         :", paired_balanced_60_config["training"]["max_epochs"])
print("S3.1 BALANCED-60 CONFIG: PASS")


In [ ]:
paired_balanced_60_summary = None
if RUN_S3_1_PAIRED_BALANCED_60:
    paired_balanced_60_summary = run_s3_experiment(
        paired_balanced_60_config,
        "s3_1_paired_ranking_balanced_init_60ep",
        paired=True,
        resume=RESUME_S3_1_PAIRED_BALANCED_60,
    )
else:
    print("Đổi RUN_S3_1_PAIRED_BALANCED_60 = True để chạy run mới.")


In [ ]:
OLD_S3_1_AUC = 0.6848095791633567
OLD_S3_1_FITB = 0.7609457092819615
FROZEN_V5_AUC = 0.6905082489625538
FROZEN_V5_FITB = 0.7626970227670753

if paired_balanced_60_summary is not None:
    metrics = paired_balanced_60_summary["best_validation_metrics"]
    print_full_metric_summary("S3.1 PAIRED BALANCED-60", paired_balanced_60_summary)
    print("AUC delta vs old S3.1 :", metrics["roc_auc"] - OLD_S3_1_AUC)
    print("FITB delta vs old S3.1:", metrics["fitb_2way"] - OLD_S3_1_FITB)
    print("AUC delta vs V5       :", metrics["roc_auc"] - FROZEN_V5_AUC)
    print("FITB delta vs V5      :", metrics["fitb_2way"] - FROZEN_V5_FITB)
else:
    print("Run chưa được thực thi; chưa có metric để so sánh.")
